# Notes
[2026-06-05] - 
Created the loading in mechanism
[2026-06-08] - 
Created the first "working" model (extremely slow, though that may be due to its genormeous size)
[2026-06-09] - changed the masking to BART-inspired one, accuracy went from 10% &rarr; 20%. Furthermore, increasing the cutoff does not improve accuracy all that much

In [1]:
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch import nn
from Bio import SeqIO 
import torch
import random
from dataclasses import dataclass
from pathlib import Path

random.seed(42)
np.random.seed (42)

print ("imported")

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu" # when running locally with newer PyTorch versions with torch.accelerator support

# device = "cuda" if torch.cuda.is_available() else "cpu" # alternative for older PyTorch versions without torch.accelerator, for use on the cluster 

print(f"Using: {device} \n")

@dataclass
class ModelArgs:
    length_cutoff: int = 100
    masking_rate: float = 0.15
    batch_size=32
    src_vocab_size = 22
    tgt_vocab_size = 22
    d_model = 128
    num_heads = 4
    num_layers = 4
    d_ff = 4*d_model
    max_seq_length = 1000
    dropout = 0.1
    num_epochs=100
    learning_rate = 1e-3
    weight_decay = 1e-3


data_path = "./data/uniprotkb_taxonomy_id_1002366_2026_06_05.fasta"

imported
Using: mps 



My plan is to separate the file into a 

In [2]:
data_path = "./data/uniprotkb_taxonomy_id_1002366_2026_06_05.fasta"

sequences = [str(record.seq) for record in SeqIO.parse(data_path, "fasta")]
sequences = [seq for seq in sequences if len(seq) < ModelArgs.length_cutoff]

print("Number of sequences:", len(sequences))

Number of sequences: 319


In [3]:
AminoAcids = sorted(list(set("".join(sequences))))  # X is a padding, - is masking
aa_vocab= ["-"] + ["X"]+ AminoAcids 
print(len(aa_vocab))
print(aa_vocab)

aa_stoi = {s: i for i, s in enumerate(aa_vocab)}
aa_itos = {i: s for i, s in enumerate(aa_vocab)}

22
['-', 'X', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']


In [4]:
traindata, valdata, testdata = torch.utils.data.random_split(sequences, [0.80, 0.10, 0.10])

print(f"Train size: {len(traindata)}, Val size: {len(valdata)}, Test size: {len(testdata)} \n")

Train size: 256, Val size: 32, Test size: 31 



In [5]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
import torch.nn.functional as F

def make_masked(seq, mask_prob=0.15):
    masked = []
    labels = []

    for aa in seq:
        if torch.rand(1).item() < mask_prob:
            labels.append(aa_stoi[aa])

            r = torch.rand(1).item()

            if r < 0.8:
                masked.append("-")
            elif r < 0.9:
                masked.append(random.choice(list(AminoAcids)))
            else:
                masked.append(aa)
        else:
            masked.append(aa)
            labels.append(-100)

    return "".join(masked), torch.tensor(labels, dtype=torch.long)

def make_collate_fn(mask_rate):
    def collate_fn(batch):
        original = list(batch)

        masked_data = [make_masked(seq, mask_rate) for seq in original]

        masked = [item[0] for item in masked_data]
        y = [item[1] for item in masked_data]

        x_ids = [torch.tensor([aa_stoi[aa] for aa in seq], dtype=torch.long) for seq in masked]

        lengths = torch.tensor([len(seq) for seq in x_ids], dtype=torch.long)

        x_ids = pad_sequence(x_ids, batch_first=True, padding_value=aa_stoi["X"])

        y = pad_sequence(y, batch_first=True, padding_value=-100)

        attention_mask = torch.arange(x_ids.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)

        return x_ids, y, attention_mask

    return collate_fn

In [6]:
train_loader = DataLoader(traindata, batch_size=ModelArgs.batch_size, shuffle=True, collate_fn=make_collate_fn(ModelArgs.masking_rate))
val_loader = DataLoader(valdata, batch_size=ModelArgs.batch_size, shuffle=False, collate_fn=make_collate_fn(ModelArgs.masking_rate))
test_loader = DataLoader(testdata, batch_size=ModelArgs.batch_size, shuffle=False, collate_fn=make_collate_fn(ModelArgs.masking_rate))

In [7]:
from bin.model import Transformer

transformer = Transformer(src_vocab_size = len(aa_vocab), tgt_vocab_size=len(aa_vocab), d_model=ModelArgs.d_model, num_heads=ModelArgs.num_heads, num_layers=ModelArgs.num_layers, d_ff=ModelArgs.d_ff, max_seq_length=ModelArgs.max_seq_length, dropout=ModelArgs.dropout)

print(transformer)

Transformer(
  (encoder_embedding): Embedding(22, 128, padding_idx=1)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (W_o): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (local_convolution): LocalConv(
        (conv): Sequential(
          (0): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(3,), groups=128)
          (1): GELU(approximate='none')
          (2): Conv1d(128, 128, kernel_size=(1,), stride=(1,))
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (norm3): LayerNor

In [9]:
from bin.train import train_model

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

transformer = train_model(
    model=transformer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    tgt_vocab_size=len(aa_vocab),
    num_epochs=ModelArgs.num_epochs,
    learning_rate=ModelArgs.learning_rate,
    save_path="./outputs/weights.pth"
    # save_path=save_path
)

NameError: name 'ModelArgs' is not defined

In [ ]:
from bin.testing import test_model

test_loss, test_accuracy = test_model(
    model=transformer,
    test_loader=test_loader,
    device=device,
    tgt_vocab_size=ModelArgs.tgt_vocab_size,
    criterion=criterion
)

Test Loss: 2.7592
Masked Test Accuracy: 17.15%
